# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/shailesh-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
"""My rule (Lane 2: Refresh / Content Opportunity)**

A page is a good refresh candidate when it has real traffic worth protecting,
it has not been updated in a long time, and its recent clicks are going down.

Score = 0.4 x staleness + 0.3 x traffic + 0.3 x slipping
- staleness: days since last update, scaled 0 to 1 (capped at 1 year)
- traffic: prior-window clicks, scaled 0 to 1 against the top 10% of pages
- slipping: 1 if prior-window click trend is negative, else 0

**Reason codes (each row gets exactly ONE):**
- STALE_AND_SLIPPING: stale (over 365 days) AND clicks trending down
- STALE_WITH_TRAFFIC: stale and has traffic, but not declining
- (add one more only if a signal check supports it)

**Action label:** REFRESH_CONTENT

**Inputs:** all from the prior window only. No future-window columns, no product flags, no labels.

**Why these signals:** staleness is the one behind FlyRank's refresh flags.
I check it and the click trend in Section 1's signal tables before trusting them.
If a check comes back OPPOSITE or FALSE, I change or drop that part of the score.
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All."""

REASON_CODES = {
    "STALE_AND_SLIPPING": "Over 365 days stale and prior clicks trending down",
    "STALE_WITH_TRAFFIC": "Over 365 days stale with traffic, not declining,"
}
ACTION_LABEL = "REFRESH_CONTENT"

for code, meaning in REASON_CODES.items():
    print(f"{code:20s} {meaning}")


STALE_AND_SLIPPING   Over 365 days stale and prior clicks trending down
STALE_WITH_TRAFFIC   Over 365 days stale with traffic, not declining,


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [55]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import numpy as np

# Safety check: Section 1 must have run first
assert "df" in globals(), "Run the Section 1 load cell first (it creates df)."

# 1) Keep rows that have all three score inputs (past window only)
d = df.dropna(subset=[STALE_COL, CLICKS_COL, TREND_COL]).copy()

# 2) Cap traffic at the 90th percentile so one huge page doesn't dominate
traffic_cap = d[CLICKS_COL].quantile(0.90)

# 3) Score = 0.4 staleness + 0.3 traffic + 0.3 slipping
d["stale_part"]    = (d[STALE_COL] / 365).clip(upper=1.0)
d["traffic_part"]  = (d[CLICKS_COL] / traffic_cap).clip(upper=1.0)
d["slipping_part"] = (d[TREND_COL] < 0).astype(float)
d["score"] = 0.4*d["stale_part"] + 0.3*d["traffic_part"] + 0.3*d["slipping_part"]

# 4) ONE reason code per row
d["reason_code"] = np.where(
    (d[STALE_COL] > 365) & (d[TREND_COL] < 0),
    "STALE_AND_SLIPPING",
    "STALE_WITH_TRAFFIC",
)

# 5) Action label
d["action"] = "REFRESH_CONTENT"

# 6) Rank everything, highest score first, and write the CSV
queue = d.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
queue[["rank", ID_COL, "score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)

print("rows written:", len(queue))
print(queue["reason_code"].value_counts())
display(queue[["rank", ID_COL, "score", "reason_code", "action"]].head(10))

AssertionError: Run the Section 1 load cell first (it creates df).

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

assert "queue" in globals(), "Run Section 2 first (it creates queue)."

N = 20   # use 10 if you only do the required top ten
top = queue.head(N).copy()

low_cut  = queue[CLICKS_COL].quantile(0.25)   # "small traffic" line
high_cut = queue[CLICKS_COL].quantile(0.75)   # "big traffic" line

# Confidence note, based on the page's own numbers
top["confidence"] = np.select(
    [
        top[CLICKS_COL] < low_cut,
        (top[STALE_COL] > 730) & (top[TREND_COL] < 0),
    ],
    [
        "LOW: small click count, trend may be noise",
        "HIGH: very stale and clearly declining",
    ],
    default="MEDIUM: one signal strong, the other weaker",
)

# Why it is here, in numbers
top["why_here"] = (
    top[STALE_COL].astype(int).astype(str) + " days stale, "
    + top[CLICKS_COL].astype(int).astype(str) + " prior clicks, trend "
    + top[TREND_COL].round(2).astype(str)
)

# What would make it wrong: depends on which signal pushed the row up
top["wrong_if"] = np.select(
    [
        top[CLICKS_COL] < low_cut,
        (top[TREND_COL] < 0) & (top[CLICKS_COL] >= high_cut),
        top[TREND_COL] < 0,
        top[STALE_COL] > 730,
    ],
    [
        "Few clicks, so the drop is only a handful of clicks and may be random noise",
        "Big page with a falling trend; the drop may be seasonal, check the same months last year",
        "Decline may come from a ranking or competitor change, not from age",
        "Old but not declining; the page may be evergreen and still fine",
    ],
    default="Score is high mostly from traffic; the page may not need a refresh at all",
)

pd.set_option("display.max_colwidth", None)
show = top[["rank", ID_COL, "action", "reason_code", "confidence", "why_here", "wrong_if"]]
display(show)

# Prints a markdown table you can paste into the markdown cell above and edit
print("\n| Rank | Action | Reason code | Confidence | Why it's here | What would make it wrong |")
print("|---|---|---|---|---|---|")
for _, r in show.iterrows():
    print(f"| {r['rank']} | {r['action']} | {r['reason_code']} | "
          f"{r['confidence'].split(':')[0]} | {r['why_here']} | {r['wrong_if']} |")


AssertionError: Run Section 2 first (it creates queue).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [64]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

for name in ["df", "queue", "top", "low_cut"]:
    assert name in globals(), f"'{name}' is missing. Run the earlier cells first (load, Section 2, Section 3)."

weak = top[top[CLICKS_COL] < low_cut]
print("Low-traffic rows in top list:", len(weak))
display(weak[["rank", ID_COL, "score", CLICKS_COL, STALE_COL, TREND_COL]])

SCORE_INPUTS = [STALE_COL, CLICKS_COL, TREND_COL]
assert OUTCOME_COL not in SCORE_INPUTS, "Outcome column leaked into the score!"
print("Score inputs:", SCORE_INPUTS)
print("Outcome used only in signal checks:", OUTCOME_COL)

AssertionError: 'df' is missing. Run the earlier cells first (load, Section 2, Section 3).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.